# 1-D pollutant diffusion/advection

We will solve:

$$ \frac{\partial C}{\partial t} +v\frac{\partial C}{\partial x} = D\frac{\partial^2C}{\partial x^2} -kC $$

where \(C(x,t)\) is pollutant concentration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# 1. MODEL PARAMETERS
# ============================================================

L = 10000          # River length (m)
T = 2 * 24 * 3600  # Simulation time = 2 days (seconds)

v = 0.20           # River velocity (m/s)
D = 10.0           # Dispersion coefficient (m^2/s)
k = 0.00001        # Pollutant decay rate (1/s)


In [ ]:
# ============================================================
# 2. NUMERICAL GRID
# ============================================================

Nx = 101           # Number of spatial points
dx = L / (Nx - 1)

# Time step
dt = 10            # seconds

Nt = int(T / dt)

x = np.linspace(0, L, Nx)

In [ ]:
# ============================================================
# 3. INITIAL CONDITION
# ============================================================

# Initially pollutant concentration is zero
C = np.zeros(Nx)

In [ ]:
# ============================================================
# 4. STORAGE FOR RESULTS
# ============================================================

# Store concentration at selected times
save_every = int(6 * 3600 / dt)  # every 6 hours

results = []
times = []

In [ ]:
# ============================================================
# 5. TIME LOOP
# ============================================================

for n in range(Nt):

    C_new = C.copy()

    # --------------------------------------------------------
    # Interior grid points
    # --------------------------------------------------------

    for i in range(1, Nx - 1):

        # Advection term
        advection = -v * (C[i] - C[i-1]) / dx

        # Diffusion term
        diffusion = D * (
            C[i+1] - 2*C[i] + C[i-1]
        ) / dx**2

        # Decay term
        decay = -k * C[i]

        # PDE
        C_new[i] = C[i] + dt * (
            advection +
            diffusion +
            decay
        )

    # --------------------------------------------------------
    # Boundary conditions
    # --------------------------------------------------------

    # Constant pollutant source at upstream boundary
    C_new[0] = 100.0

    # No pollutant entering from downstream boundary
    C_new[-1] = C_new[-2]

    # Update concentration
    C = C_new

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    if n % save_every == 0:

        results.append(C.copy())
        times.append(n * dt / 3600)  # hours


# Convert results to NumPy array
results = np.array(results)

In [ ]:
# ============================================================
# 6. CONCENTRATION PROFILE
# ============================================================

plt.figure(figsize=(10, 6))

for i in range(len(results)):

    plt.plot(
        x / 1000,
        results[i],
        label=f"{times[i]:.0f} h"
    )

plt.xlabel("Distance along river (km)")
plt.ylabel("Pollutant concentration")
plt.title("Pollutant Transport Using a PDE")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# 7. SPATIAL-TEMPORAL HEATMAP
# ============================================================

plt.figure(figsize=(10, 6))

plt.imshow(
    results,
    aspect="auto",
    extent=[
        0,
        L / 1000,
        times[-1],
        times[0]
    ],
    origin="upper"
)

plt.colorbar(label="Pollutant concentration")

plt.xlabel("Distance along river (km)")
plt.ylabel("Time (hours)")

plt.title("Spatial-Temporal Distribution of Pollutant")

plt.show()

#2-D Reaction-Diffusion PDE for Disease Risk

*   This is a more comprehensive example combining both 'reaction' (disease growth) and 'diffusion' (spatial spread) components in a 2-D Partial Differential Equation.
*   It simulates how a disease hotspot evolves over time within a spatial domain, considering both spread and intrinsic growth/decay of the disease.

*   This model uses the Finite Difference Method for its numerical solution.



Run the cell below to see a full 2-D reaction-diffusion simulation. Observe how the disease risk changes and spreads over the defined time steps.










In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ------------------------------------------------------------
# 1. Define spatial domain
# ------------------------------------------------------------

nx = 100                  # number of grid cells in X
ny = 100                  # number of grid cells in Y

Lx = 100                  # length of study area in X
Ly = 100                  # length of study area in Y

dx = Lx / (nx - 1)
dy = Ly / (ny - 1)

x = np.linspace(0, Lx, nx)
y = np.linspace(0, Ly, ny)


In [ ]:
# ------------------------------------------------------------
# 2. Define time
# ------------------------------------------------------------

dt = 0.05                 # time step
total_time = 20

nt = int(total_time / dt)

In [ ]:
# ------------------------------------------------------------
# 3. Disease model parameters
# ------------------------------------------------------------

D = 0.5                   # diffusion coefficient
r = 0.05                  # disease growth rate

In [ ]:
# ------------------------------------------------------------
# 4. Initial disease-risk distribution
# ------------------------------------------------------------

R = np.zeros((ny, nx))

# Create an initial disease hotspot
center_x = 50
center_y = 50

sigma = 8

X, Y = np.meshgrid(x, y)

R = np.exp(
    -((X - center_x)**2 + (Y - center_y)**2)
    / (2 * sigma**2)
)

In [ ]:
# ------------------------------------------------------------
# 5. Store selected time steps
# ------------------------------------------------------------

results = []

save_times = [0, 5, 10, 15, 20]


In [ ]:
# ------------------------------------------------------------
# 6. Solve PDE using finite difference method
# ------------------------------------------------------------

for n in range(nt + 1):

    current_time = n * dt

    # Save selected time steps
    if any(abs(current_time - t) < dt / 2 for t in save_times):

        results.append(
            (current_time, R.copy())
        )

    # Stop after final time
    if n == nt:
        break

    # Create copy
    R_new = R.copy()

    # --------------------------------------------------------
    # Calculate spatial diffusion
    # --------------------------------------------------------

    d2R_dx2 = (
        R[1:-1, 2:]
        - 2 * R[1:-1, 1:-1]
        + R[1:-1, :-2]
    ) / dx**2

    d2R_dy2 = (
        R[2:, 1:-1]
        - 2 * R[1:-1, 1:-1]
        + R[:-2, 1:-1]
    ) / dy**2

    # Laplacian
    laplacian = d2R_dx2 + d2R_dy2

    # --------------------------------------------------------
    # Reaction-diffusion equation
    # --------------------------------------------------------

    R_new[1:-1, 1:-1] = (
        R[1:-1, 1:-1]
        + dt * (
            D * laplacian
            + r * R[1:-1, 1:-1]
        )
    )

    # --------------------------------------------------------
    # Boundary condition
    # --------------------------------------------------------

    R_new[0, :] = 0
    R_new[-1, :] = 0
    R_new[:, 0] = 0
    R_new[:, -1] = 0

    # Update
    R = R_new

In [ ]:
# ============================================================
# 7. Plot disease-risk maps
# ============================================================

fig, axes = plt.subplots(
    1,
    len(results),
    figsize=(18, 4)
)

for ax, (time_value, risk) in zip(axes, results):

    im = ax.imshow(
        risk,
        origin="lower",
        extent=[0, Lx, 0, Ly],
        vmin=0,
        vmax=np.max(results[-1][1])
    )

    ax.set_title(
        f"Time = {time_value:.0f}"
    )

    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")

fig.colorbar(
    im,
    ax=axes,
    label="Disease Risk"
)

plt.suptitle(
    "2-D PDE Model of Disease Risk",
    fontsize=16
)

plt.tight_layout()
plt.show()
